# 🔑 Solutions — Errors — advanced concepts

One worked answer per exercise. There is usually more than one correct
way to write these — if yours runs and you can explain what it does, yours is right too.

Read this *after* an attempt: seeing the answer teaches far less than
getting stuck did.

## Our data

In [ ]:
# The genetic code: every three-letter codon -> its amino acid (one letter).
codon_table = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}
stop_codons = {"TAA", "TAG", "TGA"}

# The same short gene, sampled from two related species.
species_a = "ATGAAAGGTTTTCTGGAATATGCAGGCTAA"   # the demos use this
species_b = "ATGAAAGGCTTCCTGGAGTATGCAAGCTAA"   # your exercises use this

print("species A:", species_a)
print("species B:", species_b)


# our translate() from the functions notebooks
def translate(seq, frame=0):
    protein = ""
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        protein = protein + codon_table[codon]
    return protein

# ...and some MESSY real-world sequences to break it on
messy = "ATGAANGGTTTT"        # an 'N' - the sequencer wasn't sure of this base
lower = "atgaaaggttttctg"     # lower-case letters


# a batch of sequences - as if read from a folder of files. Some are broken.
batch = {
    "species_A":  species_a,
    "species_B":  species_b,
    "sample_03":  "ATGAANGGTTTT",       # an N base
    "sample_04":  "ATGAAAGG",           # length not a multiple of 3
    "sample_05":  "ATGTGGCATCCGTAA",    # fine
    "sample_06":  "ATG_CORRUPTED",      # junk from a bad download
}

## Solution — Skill 1: `else` and `finally`

In [ ]:
# report from the demo
def report(seq):
    try:
        protein = translate(seq)
    except KeyError:
        print("  could not translate - unknown codon")
    else:
        print("  protein:", protein)
    finally:
        print("  (finished this sequence)")

print("species B:"); report(species_b)  # else AND finally
print("lower:");     report(lower)      # except AND finally

# finally ran for BOTH. That makes it the place for clean-up that must always
# happen - closing a file, logging that we're done - whether it worked or not.

## Solution — Skill 2: your own exception types

In [ ]:
class BadLengthError(ValueError):
    """Raised when a sequence length isn't a whole number of codons."""

def check_length(seq):
    if len(seq) % 3 != 0:
        raise BadLengthError(f"{seq!r} has length {len(seq)}, not a multiple of 3")
    return seq

try:
    check_length("ATGAAAGG")
except BadLengthError as e:
    print("caught:", e)

## Solution — Skill 3: EAFP

In [ ]:
# translate_tolerant from the demo (EAFP)
def translate_tolerant(seq):
    protein = ""
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        try:
            protein = protein + codon_table[codon]
        except KeyError:
            protein = protein + "X"
    return protein

print(translate_tolerant(species_a))     # MKGFLEYAG - no X: every codon is known
print(translate_tolerant("ATGNNNTTT"))   # MXF - X only where the codon has an N

# the same, LBYL: look before you leap
def translate_lbyl(seq):
    protein = ""
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        if codon in codon_table:
            protein = protein + codon_table[codon]
        else:
            protein = protein + "X"
    return protein

print(translate_lbyl("ATGNNNTTT"))       # MXF - same result

## Solution — Skill 4: a batch job that never dies

In [ ]:
# clean_dna, NotDNAError and translate_batch from the demos
class NotDNAError(ValueError):
    """Raised when a sequence contains characters other than A/T/G/C."""

def clean_dna(seq: str) -> str:
    seq = seq.upper().strip()
    for base in seq:
        if base not in "ATGC":
            raise NotDNAError(f"{seq!r} contains {base!r}, which isn't DNA")
    return seq

def translate_batch(sequences: dict) -> dict:
    results = {}
    failures = {}
    for name, seq in sequences.items():
        try:
            clean = clean_dna(seq)
            if len(clean) % 3 != 0:
                raise BadLengthError(f"length {len(clean)} isn't a multiple of 3")
            results[name] = translate(clean)
        except ValueError as error:
            failures[name] = str(error)
    return {"results": results, "failures": failures}

batch["sample_07"] = "atgtggcattaa"      # valid: lower-case, but clean_dna fixes that
batch["sample_08"] = "ATG TGG CAT TAA"   # broken in a new way: spaces inside

report = translate_batch(batch)
print("succeeded:", len(report["results"]))   # 4
print("failed:   ", len(report["failures"]))  # 4
for name, why in report["failures"].items():
    print(f"  {name}: {why}")

# Returning both dicts means that after 100 files overnight you still have every
# good result AND one tidy list of what to fix - instead of errors scrolled away.